In [3]:
from sqlalchemy import (
    Table, MetaData, Column, Integer, String, 
    create_engine, ForeignKey, Date, Time, Enum
)

engine = create_engine('postgresql:///premier_league')

metadata = MetaData()

competition_table = Table(
    'competition', metadata,
    Column('idcompetition', Integer, primary_key=True),
    Column('nomcompetition', String(155))
)

saison_table = Table(
    'saison', metadata,
    Column('id_saison', Integer, primary_key=True),
    Column('annee', Integer)
)

equipe_table = Table(
    'equipe', metadata,
    Column('idequipe', Integer, primary_key=True),
    Column('nomequipe', String(250)),
    Column('idcompetition', Integer, ForeignKey('competition.idcompetition')),
    Column('idsaison', Integer, ForeignKey('saison.id_saison'))
)

joueur_table = Table(
    'joueur', metadata,
    Column('idjoueur', Integer, primary_key=True),
    Column('nomjoueur', String(250)),
    Column('position', String(100)),
    Column('nationalite', String(100)),
    Column('id_equipe', Integer, ForeignKey('equipe.idequipe'))
)

match_table = Table(
    'match', metadata,
    Column('idmatch', Integer, primary_key=True),
    Column('date_match', Date),
    Column('heure', Time),
    Column('round', String(100)),
    Column('venue', String(250)),
    Column('idteamhome', Integer, ForeignKey('equipe.idequipe')),
    Column('idteam_away', Integer, ForeignKey('equipe.idequipe')),
    Column('id_competition', Integer, ForeignKey('competition.idcompetition')),
    Column('id_saison', Integer, ForeignKey('saison.id_saison'))
)

resultatmatch_table = Table(
    'resultatmatch', metadata,
    Column('idresultat', Integer, primary_key=True),
    Column('idmatch', Integer, ForeignKey('match.idmatch')),
    Column('idequipe', Integer, ForeignKey('equipe.idequipe')),
    Column('butsmarques', Integer),
    Column('butsconcedes', Integer),
    Column('resultat', Enum('Victoire', 'Défaite', 'Nul', name='resultat_enum', create_type=True))
)

statistiquejoueur_table = Table(
    'statistiquejoueur', metadata,
    Column('idstats', Integer, primary_key=True),
    Column('idjoueur', Integer, ForeignKey('joueur.idjoueur')),
    Column('buts', Integer, default=0),
    Column('passesdecisives', Integer, default=0),
    Column('nbmatchesplayed', Integer, default=0),
    Column('cartonsjaunes', Integer, default=0),
    Column('cartonsrouges', Integer, default=0)
)

if __name__ == "__main__":
    try:
        metadata.create_all(engine)
        print("success")
    except Exception as e:
        print(f"error: {e}")
        

success


In [29]:
from sqlalchemy import insert, select
from sqlalchemy import create_engine
import pandas as pd


engine = create_engine('postgresql:///premier_league')
df_matchs = pd.read_csv('premier_league_matchs.csv')

with engine.begin() as conn:
    competitions = df_matchs['comp'].unique()
    comp_dict = {}

    for i, name in enumerate(competitions, start=1):
        conn.execute(insert(competition_table).values(idcompetition=i, nomcompetition=name))
        comp_dict[name] = i

    conn.execute(insert(saison_table).values(id_saison=1, annee=2024))

    # all_teams = sorted(set(df_matchs['team']).union(df_matchs['opponent']))
    # premier_league_id = comp_dict.get('Premier League', 1)

    # equipes_data = [
    #     dict(idequipe=i, nomequipe=team, idcompetition=premier_league_id, idsaison=1)
    #     for i, team in enumerate(all_teams, start=1)
    # ]
    # conn.execute(insert(equipe_table), equipes_data)

    print(f"inserted: {len(competitions)} competitions, 1 saison, equipes")
    for c in competitions:
        print(f"  - {c}")



inserted: 7 competitions, 1 saison, equipes
  - Premier League
  - EFL Cup
  - Champions Lg
  - FA Cup
  - FA Community Shield
  - Conf Lg
  - Europa Lg


In [30]:

df_matchs_opp_comp = df_matchs[['opponent', 'comp']].drop_duplicates()
df_matchs_team_comp = df_matchs[['team', 'comp']].drop_duplicates()
df_matchs_opp_comp = df_matchs_opp_comp.rename(columns={'opponent': 'team'})
all_team_comps = pd.concat([df_matchs_team_comp, df_matchs_opp_comp]).drop_duplicates()
team_to_comp_map = all_team_comps.groupby('team')['comp'].first().to_dict()


equipes_data = [] 
all_teams = sorted(set(df_matchs['team']).union(df_matchs['opponent']))

with engine.begin() as conn:
    

    for idx, team_name in enumerate(all_teams):
        comp_name = team_to_comp_map.get(team_name)
            
        comp_id = None
        if comp_name:
            comp_id_query = select(competition_table.c.idcompetition).where(
                competition_table.c.nomcompetition == comp_name
            )
            comp_id = conn.execute(comp_id_query).scalar()
        else : 
            print(f'error: competition not found for team {team_name}')
        
        
        equipes_data.append(
            dict(idequipe=idx, nomequipe=team_name, idcompetition=comp_id, idsaison=1)
        )
        
 
    conn.execute(insert(equipe_table), equipes_data)

    print("Insertion complete.")

Insertion complete.


In [28]:
df_matchs

,date,start_time,comp,round,dayofweek,venue,result,goals_for,goals_against,opponent,xg_for,xg_against,possession,attendance,captain,formation,opp_formation,referee,team
0,2024-08-25,16:30,Premier League,Matchweek 2,Sun,Home,W,2,0,Brentford,2.5,0.5,62.0,"60,017",Virgil van Dijk,4-2-3-1,4-4-2,Stuart Attwell,Liverpool
1,2024-09-14,15:00,Premier League,Matchweek 4,Sat,Home,L,0,1,Nott'ham Forest,0.9,0.4,68.0,"60,344",Virgil van Dijk,4-2-3-1,4-2-3-1,Michael Oliver,Liverpool
2,2024-09-21,15:00,Premier League,Matchweek 5,Sat,Home,W,3,0,Bournemouth,2.0,1.1,58.0,"60,347",Virgil van Dijk,4-2-3-1,4-2-3-1,Tony Harrington,Liverpool
3,2024-09-25,20:00,EFL Cup,Third round,Wed,Home,W,5,1,West Ham,NaN,NaN,61.0,"60,044",Joe Gomez,4-2-3-1,4-2-3-1,Andy Madley,Liverpool
4,2024-10-02,20:00,Champions Lg,League phase,Wed,Home,W,2,0,it Bologna,1.2,0.6,51.0,"59,816",Virgil van Dijk,4-2-3-1,4-1-4-1,Nikola Dabanović,Liverpool
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,2025-04-02,19:45 (18:45),Premier League,Matchweek 30,Wed,Home,D,1,1,Crystal Palace,0.7,0.7,43.0,"30,158",Jack Stephens,3-4-3,3-4-3,Andy Madley,Southampton
484,2025-04-12,15:00,Premier League,Matchweek 32,Sat,Home,L,0,3,Aston Villa,0.3,3.0,40.0,"30,199",Jack Stephens,3-4-3,4-2-3-1,Thomas Bramall,Southampton
485,2025-04-26,15:00,Premier League,Matchweek 34,Sat,Home,L,1,2,Fulham,0.6,2.4,35.0,"28,946",Jack Stephens,3-4-3,4-3-3,Tony Harrington,Southampton
486,2025-05-10,15:00,Premier League,Matchweek 36,Sat,Home,D,0,0,Manchester City,0.1,1.7,28.0,"30,937",Jack Stephens,3-4-3,4-2-3-1,Tim Robinson,Southampton
